In [1]:
import pandas as pd

CONFIG_FILES = [
    "setup.cfg",
    "pyproject.toml",
    "setup.py",
    "requirements.txt",
    "pom.xml",
]

updates_df = {}
for config_file in CONFIG_FILES:
    updates_df[config_file] = pd.read_csv(
        f"benchmark/updates/{config_file}_updates.csv",
        low_memory=False,
        keep_default_na=False,
    )

In [2]:
def count_releases():
    result = []
    for config_file in CONFIG_FILES:
        commit_df = pd.read_csv(
            f"benchmark/commits/{config_file}_commits.csv",
            low_memory=False,
            keep_default_na=False,
        )
        num_modify_commits = len(commit_df)
        num_blobs = pd.concat([commit_df["new blob"], commit_df["old blob"]]).nunique()

        update = updates_df[config_file]
        num_update_commits = update["commit"].nunique()
        num_updates = len(update)
        num_pkgs = update["package"].nunique()
        df1 = update[["package", "version before"]].rename(
            columns={"version before": "version"}
        )
        df2 = update[["package", "version after"]].rename(
            columns={"version after": "version"}
        )
        num_releases = len(pd.concat([df1, df2], ignore_index=True).drop_duplicates())
        result.append(
            [
                config_file,
                num_modify_commits,
                num_blobs,
                num_update_commits,
                num_updates,
                num_pkgs,
                num_releases,
            ]
        )

    result_df = pd.DataFrame(
        result,
        columns=[
            "Configuration File",
            "#ModifyCommits",
            "#UniqueBlobs",
            "#UpdateCommits",
            "#Updates",
            "#Packages",
            "#Releases",
        ],
    )
    return result_df


count_releases()

,Configuration File,#ModifyCommits,#UniqueBlobs,#UpdateCommits,#Updates,#Packages,#Releases
0,setup.cfg,1229076,719294,29832,64098,1976,14333
1,pyproject.toml,2956927,2045404,64965,103115,2623,18026
2,setup.py,8528862,5525611,238418,366487,8154,84926
3,requirements.txt,17264495,10189499,7719071,13473774,31419,340392
4,pom.xml,97785879,63959850,6103952,15314119,67923,1105308


In [3]:
def update_type(row):
    result = ["Major", "Minor", "Patch"]
    ver_before = row["version before"].split(".")
    ver_after = row["version after"].split(".")
    if (ver_before[0] == "0") and (ver_after[0] == "0"):
        return "Dev"
    min_len = min(len(ver_before), len(ver_after))
    for i in range(min_len):
        if ver_before[i] != ver_after[i]:
            return result[i]
    return result[min_len]

In [4]:
def count_update_type():
    dfs = []
    for config_file, update_df in updates_df.items():
        tmp = update_df.copy()
        tmp["config file"] = config_file
        tmp["update type"] = tmp.apply(update_type, axis=1)
        dfs.append(tmp)
    return pd.concat(dfs, ignore_index=True)

In [ ]:
all_updates = count_update_type()
all_updates[
    [
        "commit",
        "filepath",
        "package",
        "version before",
        "version after",
        "config file",
        "update type",
    ]
].to_csv("benchmark/updates/c2fpkgvvtype.csv", index=False)

In [ ]:
update_type_dist = (
    all_updates.groupby(["config file", "update type"])
    .size()
    .reset_index(name="count")
    .pivot_table(
        values="count", index="config file", columns="update type", fill_value=0
    )
    .astype(int)
    .rename_axis(columns=None, index=None)
    .reindex(CONFIG_FILES)[["Major", "Minor", "Patch", "Dev"]]
)
update_type_dist

,Major,Minor,Patch,Dev
setup.cfg,8311,23648,16873,15266
pyproject.toml,12640,31306,27741,31428
setup.py,60605,128781,94377,82724
requirements.txt,2934473,5699163,3242361,1597777
pom.xml,1802688,6744627,6208775,558029


In [ ]:
update_type_dist.div(update_type_dist.sum(axis=1), axis=0)

,Major,Minor,Patch,Dev
setup.cfg,0.129661,0.368935,0.263238,0.238167
pyproject.toml,0.122582,0.303603,0.269030,0.304786
setup.py,0.165367,0.351393,0.257518,0.225722
requirements.txt,0.217791,0.422982,0.240642,0.118584
pom.xml,0.117714,0.440419,0.405428,0.036439
